# Step 5 - Analysis of bicycle network results
## Project: Growing Urban Bicycle Networks with LTNs

This notebook takes the existing infrastructure, the results and makes some extra plots for the paper

TODO


In [1]:
# import libraries
from src import utils
PATH = utils.PATH # shortening the var name so that we don't have to change it below

# System
import csv
import os
import dill as pickle
import itertools
import random
from collections import defaultdict
import pprint
pp = pprint.PrettyPrinter(indent=4)
from tqdm.notebook import tqdm
import glob
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
import yaml
import json

# Maths/Data
import numpy as np
import pandas as pd
import math


# Network
import networkx as nx

# Plotting
import matplotlib.pyplot as plt
import matplotlib.animation as animation


# Geo
import osmnx as ox
ox.settings.log_file = True
ox.settings.requests_timeout = 300
ox.settings.logs_folder = PATH["logs"]
import geopandas as gpd
import json


## Preliminaries

### Parameters

In [2]:
debug = False # If True, will produce plots and/or verbose output to double-check
# if not debug: # Only do this if sure the code is bug-free!
#     warnings.filterwarnings('ignore')
rerun_existing = True # If True, will re-run the costly analysis of existing infra even if files already exist.
rerun = False # If True, recompute the analysis. If false, just re-make the plots

In [3]:
params = yaml.load(
    open("../parameters/parameters.yml"), 
    Loader=yaml.FullLoader)
osmnxparameters = json.load(open("../parameters/osmnxparameters.json", "r"))
plotparam = json.load(open("../parameters/plotparam.json", "r"))
plotparam_analysis = json.load(open("../parameters/plotparam_analysis.json", "r"))

### Network weighting by tags

In [4]:
tag_lts = json.load(open("../parameters/tag_lts.json", "r"))
distance_cost = json.load(open("../parameters/distance_cost.json", "r"))
lts_class = json.load(open("../parameters/lts_class.json", "r"))

### Load Cities

In [5]:
# load cities
#cities = utils.load_cities(PATH, debug)

# temporary: do it manually
cities = {
    "newcastle": {"nominatimstring": "Newcastle Upon Tyne", "countryid": "gbr", "name": "Newcastle Upon Tyne"},
    "sunderland": {"nominatimstring": "Sunderland", "countryid": "gbr", "name": "Sunderland"},
    "north_tyneside": {"nominatimstring": "North Tyneside", "countryid": "gbr", "name": "North Tyneside"},
    "south_tyneside": {"nominatimstring": "South Tyneside", "countryid": "gbr", "name": "South Tyneside"},
    "gateshead": {"nominatimstring": "Gateshead", "countryid": "gbr", "name": "Gateshead"}
}


## Loading

### Load Results

In [6]:
# betweenness 
betweenness_results = {}
for scenario in params["scenarios"]:
    betweenness_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_betweenness_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                betweenness_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the betweenness analysis first.")
            print(f"No betweenness files found for {placeid} in scenario {scenario}.")

In [7]:
# random (many runs to get a distribution)
random_results = {}
for scenario in params["scenarios"]:
    random_results[scenario] = {}
    for placeid in cities:
        pattern = (PATH["results"] + placeid + "/" + scenario + "/" +
                   f"{placeid}_poi_{params['poi_source']}_random_weighted_{scenario}_run*.pickle")
        random_files = sorted(glob.glob(os.path.abspath(pattern)))[:1]  # only take the first one for this plotting
        if random_files:
            random_results[scenario][placeid] = []
            for fn in random_files:
                abs_path = os.path.abspath(fn)
                with open(abs_path, "rb") as f:
                    res = pickle.load(f)
                random_results[scenario][placeid].append(res)
        else:
            print(f"No random files found for {placeid} in scenario {scenario}.")
            print("Please run the random growth analysis first.")


In [8]:
# demand
demand_results = {}
for scenario in params["scenarios"]:
    demand_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_demand_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                demand_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the demand analysis first.")
            print(f"No demand files found for {placeid} in scenario {scenario}.")


In [9]:
# demand LTN priority
demand_ltn_priority_results = {}
for scenario in params["scenarios"]:
    demand_ltn_priority_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_demand_ltn_priority_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                demand_ltn_priority_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the demand LTN priority analysis first.")
            print(f"No demand LTN priority files found for {placeid} in scenario {scenario}.")


File c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\sunderland\no_ltn_scenario\sunderland_poi_LTNs_tessellation_demand_ltn_priority_weighted_no_ltn_scenario.pickle does not exist.
Please run the demand LTN priority analysis first.
No demand LTN priority files found for sunderland in scenario no_ltn_scenario.
File c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\north_tyneside\no_ltn_scenario\north_tyneside_poi_LTNs_tessellation_demand_ltn_priority_weighted_no_ltn_scenario.pickle does not exist.
Please run the demand LTN priority analysis first.
No demand LTN priority files found for north_tyneside in scenario no_ltn_scenario.
File c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\south_tyneside\no_ltn_scenario\south_tyneside_poi_LTNs_tessellation_demand_ltn_priority_weighted_

In [10]:
# betweenness LTN priority
betweenness_ltn_priority_results = {}
for scenario in params["scenarios"]:
    betweenness_ltn_priority_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_betweenness_ltn_priority_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                betweenness_ltn_priority_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the betweenness LTN priority analysis first.")
            print(f"No betweenness LTN priority files found for {placeid} in scenario {scenario}.")


File c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\sunderland\no_ltn_scenario\sunderland_poi_LTNs_tessellation_betweenness_ltn_priority_weighted_no_ltn_scenario.pickle does not exist.
Please run the betweenness LTN priority analysis first.
No betweenness LTN priority files found for sunderland in scenario no_ltn_scenario.
File c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\north_tyneside\no_ltn_scenario\north_tyneside_poi_LTNs_tessellation_betweenness_ltn_priority_weighted_no_ltn_scenario.pickle does not exist.
Please run the betweenness LTN priority analysis first.
No betweenness LTN priority files found for north_tyneside in scenario no_ltn_scenario.
File c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\networkGrowth\unforked\bikenwgrowth_external\results\south_tyneside\no_ltn_scenario\south_tyneside_poi_LTNs_tessellation

Find investment level, split results into GTs, GT_abstracts 

In [11]:
for scenario_name in params["scenarios"]:
    for placeid in cities:

        # Demand 
        if placeid in demand_results.get(scenario_name, {}):
            demand_dict = demand_results[scenario_name][placeid]
            investment_levels_demand = demand_dict["prune_quantiles"]
            GTs_demand               = demand_dict["GTs"]
            GT_abstracts_demand      = demand_dict["GT_abstracts"]
        else:
            print(f"No demand results for {placeid} in scenario '{scenario_name}'")
            investment_levels_demand = []
            GTs_demand               = []
            GT_abstracts_demand      = []


        # Betweenness‐LTN‐priority 
        if placeid in betweenness_ltn_priority_results.get(scenario_name, {}):
            betweenness_ltn_dict = betweenness_ltn_priority_results[scenario_name][placeid]
            investment_levels_betw = betweenness_ltn_dict["prune_quantiles"]
            GTs_betw               = betweenness_ltn_dict["GTs"]
            GT_abstracts_betw      = betweenness_ltn_dict["GT_abstracts"]
        else:
            # e.g. scenario == "no_ltn_scenario" has no betweenness‐LTN‐priority data
            investment_levels_betw = []
            GTs_betw               = []
            GT_abstracts_betw      = []

        # Betweenness
        if placeid in betweenness_results.get(scenario_name, {}):
            betweenness_dict = betweenness_results[scenario_name][placeid]
            investment_levels_betweenness = betweenness_dict["prune_quantiles"]
            GTs_betweenness               = betweenness_dict["GTs"]
            GT_abstracts_betweenness      = betweenness_dict["GT_abstracts"]
        else:
            investment_levels_betweenness = []
            GTs_betweenness               = []
            GT_abstracts_betweenness      = []

        # Demand‐LTN‐priority 
        if placeid in demand_ltn_priority_results.get(scenario_name, {}):
            dem_ltn_dict = demand_ltn_priority_results[scenario_name][placeid]
            investment_levels_dem_ltn = dem_ltn_dict["prune_quantiles"]
            GTs_dem_ltn               = dem_ltn_dict["GTs"]
            GT_abstracts_dem_ltn      = dem_ltn_dict["GT_abstracts"]
        else:
            investment_levels_dem_ltn = []
            GTs_dem_ltn               = []
            GT_abstracts_dem_ltn      = []

        # Random‐runs (loads all run*.pickle files)
        random_runs_list = random_results.get(scenario_name, {}).get(placeid, [])
        if random_runs_list:
            all_GTs_random       = [run_dict["GTs"]          for run_dict in random_runs_list]
            all_GTabs_random      = [run_dict["GT_abstracts"]  for run_dict in random_runs_list]
            investment_levels_random = random_runs_list[0]["prune_quantiles"]
        else:
            all_GTs_random          = []
            all_GTabs_random         = []
            investment_levels_random = []



In [12]:
import networkx as nx

def ensure_crs(G, assumed_crs="EPSG:3857", target_crs="EPSG:4326"):
    """
    Ensure a MultiDiGraph has CRS metadata set.
    - If missing, assume assumed_crs (default EPSG:3857).
    - Does not reproject the graph itself, only tags it.
    Reprojection should be done after conversion to GeoDataFrame.
    """
    if G is None:
        return G
    if not isinstance(G, nx.MultiDiGraph):
        return G  # skip anything unexpected
    
    if "crs" not in G.graph or G.graph["crs"] is None:
        G.graph["crs"] = assumed_crs
    return G

for scenario_name in params["scenarios"]:
    for placeid in cities:
        GT_abstracts_demand      = [ensure_crs(g) for g in demand_dict["GT_abstracts"]]
        GT_abstracts_betw        = [ensure_crs(g) for g in betweenness_ltn_dict["GT_abstracts"]]
        GT_abstracts_betweenness = [ensure_crs(g) for g in betweenness_dict["GT_abstracts"]]
        GT_abstracts_dem_ltn     = [ensure_crs(g) for g in dem_ltn_dict["GT_abstracts"]]
        all_GTabs_random         = [[ensure_crs(g) for g in run_dict["GT_abstracts"]] for run_dict in random_runs_list]




### Load existing networks, nodes, GeoDataframe



In [13]:
G_biketracks_dict               = {}  # (placeid, scenario) → biketrack graph
G_biketrack_no_ltns_dict       = {}  # (placeid, scenario) → biketrack_no_ltn graph
G_biketrackcaralls_dict        = {}  # (placeid, scenario) → biketrackcarall graph
G_biketrackcarall_edges_dict    = {}  # (placeid, scenario) → GeoDataFrame of biketrackcarall edges
boundary_gdfs               = {}  # placeid → boundary GeoDataFrame (same for all scenarios)
tess_points_dict            = {}  # (placeid, scenario) → tessellation points GeoDataFrame
ltn_points_dict             = {}  # (placeid, scenario) → LTN points GeoDataFrame
combined_points_dict        = {}  # (placeid, scenario) → combined points GeoDataFrame
ltns_dict                  = {}  # (placeid, scenario) → LTN GeoDataFrame

for scenario in params["scenarios"]:
    for placeid, placeinfo in cities.items():
        base_folder = os.path.join(PATH["data"], placeid, scenario)

        # Load biketrack graph
        biketrack_gpkg = os.path.join(base_folder, f"{placeid}_biketrack.gpkg")
        if os.path.exists(biketrack_gpkg):
            G_biketrack = utils.ox_gpkg_to_graph(biketrack_gpkg)
            G_biketrack.remove_nodes_from(list(nx.isolates(G_biketrack)))
            G_biketracks_dict[(placeid, scenario)] = G_biketrack
        else:
            print(f"Missing: {biketrack_gpkg}")
            G_biketracks_dict[(placeid, scenario)] = None

        # Load biketrack_no_ltn graph
        biketrack_no_ltn_gpkg = os.path.join(base_folder, f"{placeid}_biketrack_no_ltn.gpkg")
        if os.path.exists(biketrack_no_ltn_gpkg):
            G_no_ltn = utils.ox_gpkg_to_graph(biketrack_no_ltn_gpkg)
            G_no_ltn.remove_nodes_from(list(nx.isolates(G_no_ltn)))
            G_biketrack_no_ltns_dict[(placeid, scenario)] = G_no_ltn
        else:
            print(f"Missing: {biketrack_no_ltn_gpkg}")
            G_biketrack_no_ltns_dict[(placeid, scenario)] = None

        # Load biketrackcarall graph
        biketrackcarall_gpkg = os.path.join(base_folder, f"{placeid}_biketrackcarall.gpkg")
        if os.path.exists(biketrackcarall_gpkg):
            G_carall = utils.ox_gpkg_to_graph(biketrackcarall_gpkg)
            G_carall.remove_nodes_from(list(nx.isolates(G_carall)))
            G_biketrackcaralls_dict[(placeid, scenario)] = G_carall

            # also store edges GeoDataFrame
            edges_gdf = ox.graph_to_gdfs(G_carall, nodes=False)
            G_biketrackcarall_edges_dict[(placeid, scenario)] = edges_gdf
        else:
            print(f"Missing: {biketrackcarall_gpkg}")
            G_biketrackcaralls_dict[(placeid, scenario)] = None
            G_biketrackcarall_edges_dict[(placeid, scenario)] = None

        #  Load boundary once per placeid (it won’t change by scenario)
        if placeid not in boundary_gdfs:
            boundary_gdf = ox.geocode_to_gdf(placeinfo["nominatimstring"])
            boundary_gdfs[placeid] = boundary_gdf

        # get nodes
        tess_points_gpkg = os.path.join(base_folder, f"{placeid}_tessellation_points.gpkg")
        if os.path.exists(tess_points_gpkg):
            tess_points = gpd.read_file(tess_points_gpkg)
            tess_points_dict[(placeid, scenario)] = tess_points
        else:
            print(f"Missing: {tess_points_gpkg}")
            tess_points_dict[(placeid, scenario)] = None
        
        # get ltn points
        if scenario != "no_ltn_scenario":
            ltn_points_gpkg = os.path.join(base_folder, f"{placeid}_ltn_points.gpkg")
            if os.path.exists(ltn_points_gpkg):
                ltn_points = gpd.read_file(ltn_points_gpkg)
                ltn_points_dict[(placeid, scenario)] = ltn_points
            else:
                print(f"Missing: {ltn_points_gpkg}")
                ltn_points_dict[(placeid, scenario)] = None
        
        # get combined points
        combined_points_gpkg = os.path.join(base_folder, f"{placeid}_combined_points.gpkg")
        if os.path.exists(combined_points_gpkg):
            combined_points = gpd.read_file(combined_points_gpkg)
            combined_points_dict[(placeid, scenario)] = combined_points
        else:
            print(f"Missing: {combined_points_gpkg}")
            combined_points_dict[(placeid, scenario)] = None

        # get all neighbourhoods (ragardless of their low traffic status. This doesn't change by scenario)
        all_neighbourhoods = gpd.read_file(PATH["data"] + placeid + "/" + 'neighbourhoods_'+  placeid + '.gpkg')
        all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid
        all_neighbourhoods_centroids = gpd.GeoDataFrame(geometry= all_neighbourhoods_centroids, crs=all_neighbourhoods.crs)

        # get ltns
        if scenario != "no_ltn_scenario":
            ltns = gpd.read_file(PATH["data"] + placeid + "/" + scenario + "/" + 'scored_neighbourhoods_' + placeid + '.gpkg')
            ltns_dict[(placeid, scenario)] = ltns

Missing: ../../bikenwgrowth_external/data/newcastle\no_ltn_scenario\newcastle_tessellation_points.gpkg


C:\Users\b8008458\AppData\Local\Temp\ipykernel_11016\1489041164.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid


Missing: ../../bikenwgrowth_external/data/sunderland\no_ltn_scenario\sunderland_tessellation_points.gpkg
Missing: ../../bikenwgrowth_external/data/sunderland\no_ltn_scenario\sunderland_combined_points.gpkg


C:\Users\b8008458\AppData\Local\Temp\ipykernel_11016\1489041164.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid


Missing: ../../bikenwgrowth_external/data/north_tyneside\no_ltn_scenario\north_tyneside_tessellation_points.gpkg


...

C:\Users\b8008458\AppData\Local\Temp\ipykernel_11016\1489041164.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid


Missing: ../../bikenwgrowth_external/data/south_tyneside\more_ltn_scenario\south_tyneside_tessellation_points.gpkg


C:\Users\b8008458\AppData\Local\Temp\ipykernel_11016\1489041164.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid


Missing: ../../bikenwgrowth_external/data/gateshead\more_ltn_scenario\gateshead_tessellation_points.gpkg


C:\Users\b8008458\AppData\Local\Temp\ipykernel_11016\1489041164.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid


In [14]:
# setup

def csv_to_ox(p, placeid, parameterid):
    '''
    Load graph from csv files (nodes and edge)
    Include OSMID, length, highway, x, y attributes
    '''

    prefix = placeid + '_' + parameterid
    compress = utils.check_extract_zip(p, prefix)
    
    with open(p + prefix + '_edges.csv', 'r') as f:
        header = f.readline().strip().split(",")
        lines = []
        for line in csv.reader(f, quotechar='"', delimiter=',', quoting=csv.QUOTE_ALL, skipinitialspace=True):
            line_list = [c for c in line]
            osmid = str(eval(line_list[header.index("osmid")])[0]) if isinstance(eval(line_list[header.index("osmid")]), list) else line_list[header.index("osmid")]
            length = str(eval(line_list[header.index("length")])[0]) if isinstance(eval(line_list[header.index("length")]), list) else line_list[header.index("length")]
            highway = line_list[header.index("highway")]
            if highway.startswith("[") and highway.endswith("]"):
                highway = highway.strip("[]").split(",")[0].strip(" '")
            line_string = f"{line_list[header.index('u')]} {line_list[header.index('v')]} {osmid} {length} {highway}"
            lines.append(line_string)
        G = nx.parse_edgelist(lines, nodetype=int, data=(("osmid", int), ("length", float), ("highway", str)), create_using=nx.MultiDiGraph)
    
    with open(p + prefix + '_nodes.csv', 'r') as f:
        header = f.readline().strip().split(",")
        values_x = {}
        values_y = {}
        for line in csv.reader(f, quotechar='"', delimiter=',', quoting=csv.QUOTE_ALL, skipinitialspace=True):
            line_list = [c for c in line]
            osmid = int(line_list[header.index("osmid")])
            values_x[osmid] = float(line_list[header.index("x")])
            values_y[osmid] = float(line_list[header.index("y")])
        nx.set_node_attributes(G, values_x, "x")
        nx.set_node_attributes(G, values_y, "y")
    
    if compress:
        os.remove(p + prefix + '_nodes.csv')
        os.remove(p + prefix + '_edges.csv')
    return G



# Plot each city at intervals

In [15]:

ltns_dict                  = {}  # (placeid, scenario) → LTN GeoDataFrame

for scenario in params["scenarios"]:
    for placeid, placeinfo in cities.items():
        base_folder = os.path.join(PATH["data"], placeid, scenario)
# get ltns
        if scenario != "no_ltn_scenario":
            ltns = gpd.read_file(PATH["data"] + placeid + "/" + scenario + "/" + 'scored_neighbourhoods_' + placeid + '.gpkg')
            ltns_dict[(placeid, scenario)] = ltns

Plotting at 0%, 1%, 25%, 50%, 75% and 100%

In [19]:
percent_labels = ["1%", "5%", "10%", "25%", "50%", "75%", "100%"]

def ensure_graph_crs(G, assumed_crs="EPSG:3857"):
    if G is None or not isinstance(G, nx.MultiDiGraph):
        return G
    if "crs" not in G.graph or G.graph["crs"] is None:
        G.graph["crs"] = assumed_crs
    return G

for scenario_name in params["scenarios"]:
    for placeid in cities:

        boundary_gdf = boundary_gdfs.get(placeid)
        if boundary_gdf is None:
            continue

        demand_dict     = demand_results.get(scenario_name, {}).get(placeid, {})
        betw_dict       = betweenness_results.get(scenario_name, {}).get(placeid, {})
        betw_ltn_dict   = betweenness_ltn_priority_results.get(scenario_name, {}).get(placeid, {})
        demand_ltn_dict = demand_ltn_priority_results.get(scenario_name, {}).get(placeid, {})
        random_runs     = random_results.get(scenario_name, {}).get(placeid, [])

        methods = {"demand": {"GTs": demand_dict.get("GTs", []), "GT_abstracts": [ensure_graph_crs(g) for g in demand_dict.get("GT_abstracts", [])]},
            "betweenness": {"GTs": betw_dict.get("GTs", []), "GT_abstracts": [ensure_graph_crs(g) for g in betw_dict.get("GT_abstracts", [])]},
            "betweenness_ltn": {"GTs": betw_ltn_dict.get("GTs", []), "GT_abstracts": [ensure_graph_crs(g) for g in betw_ltn_dict.get("GT_abstracts", [])]},
            "demand_ltn": {"GTs": demand_ltn_dict.get("GTs", []), "GT_abstracts": [ensure_graph_crs(g) for g in demand_ltn_dict.get("GT_abstracts", [])]},
            "random": {"GTs": random_runs[0].get("GTs", []) if random_runs else [],
                       "GT_abstracts": [ensure_graph_crs(g) for g in random_runs[0].get("GT_abstracts", [])] if random_runs else []} }

        if scenario_name == "no_ltn_scenario":
            methods = {k: v for k, v in methods.items() if not k.endswith("_ltn")}

        for method_name, data in methods.items():
            for network_type, GT_list in data.items():
                if not GT_list:
                    continue

                n = len(GT_list)
                idxs = [i for i in [0, 4, 9, 24, 49, 74, n-1] if 0 <= i < n]
                valid_idxs = [i for i in idxs if GT_list[i] is not None]
                if not valid_idxs:
                    continue

                fig, axes = plt.subplots(1, len(valid_idxs), figsize=(5 * len(valid_idxs), 5))
                if len(valid_idxs) == 1:
                    axes = [axes]

                for ax, idx, label in zip(axes, sorted(valid_idxs), percent_labels[:len(valid_idxs)]):
                    G = GT_list[idx]

                    boundary_gdf.to_crs("EPSG:4326").plot(ax=ax, facecolor="lightgrey", edgecolor="none")

                    road_edges_gdf = G_biketrackcarall_edges_dict.get((placeid, scenario_name))
                    if road_edges_gdf is not None and not road_edges_gdf.empty:
                        road_edges_gdf.to_crs("EPSG:4326").plot(ax=ax, color="black", linewidth=0.5, alpha=0.5)

                    if G.number_of_edges() > 0:
                        edges_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
                        if edges_gdf.crs is None:
                            edges_gdf.set_crs("EPSG:3857", inplace=True)
                        edges_gdf = edges_gdf.to_crs(road_edges_gdf.crs if road_edges_gdf is not None else "EPSG:4326")
                        if not edges_gdf.empty and edges_gdf.geometry.notnull().any():
                            edges_gdf.plot(ax=ax, linewidth=1.2, color="orange" if network_type == "GTs" else "red")

                    if network_type == "GTs":
                        biketrack_edges_g = G_biketracks_dict.get((placeid, scenario_name))
                        if biketrack_edges_g is not None:
                            edges_gdf_bt = ox.graph_to_gdfs(biketrack_edges_g, nodes=False, edges=True)
                            if not edges_gdf_bt.empty and edges_gdf_bt.geometry.notnull().any():
                                edges_gdf_bt.to_crs("EPSG:4326").plot(ax=ax, color="green", linewidth=1)

                        # --- LTN overlay: slightly transparent fill with blue outline ---
                        ltns = ltns_dict.get((placeid, scenario_name))
                        if ltns is not None and not ltns.empty:
                            try:
                                ltns_plot = ltns.to_crs("EPSG:4326")
                            except Exception:
                                ltns_plot = ltns.set_crs(boundary_gdf.crs, allow_override=True).to_crs("EPSG:4326")

                            # Plot ensure it’s on top
                            # transparent fill
                            ltns_plot.plot(
                                ax=ax,
                                facecolor='lightblue',
                                edgecolor='none',
                                alpha=0.4,        # transparent fill
                                zorder=998)

                            # darker edges
                            ltns_plot.plot(
                                ax=ax,
                                facecolor='none',
                                edgecolor='darkblue',
                                linewidth=1.5,
                                alpha=0.4,        # more visible outline
                                zorder=999)
                                    
                                                        
                    ax.set_title(label)
                    ax.axis("off")

                fig.suptitle(f"{placeid} | {scenario_name} | {method_name} | {network_type}", fontsize=16)
                plt.tight_layout(rect=[0, 0, 1, 0.95])

                save_dir = os.path.join(PATH['plots'], placeid, scenario_name)
                os.makedirs(save_dir, exist_ok=True)

                # distinguish abstract vs routed GTs in filename
                suffix = "_abstract" if network_type == "GT_abstracts" else ""
                save_fp = os.path.join(save_dir, f"{method_name}{suffix}_growth.png")
                fig.savefig(save_fp, dpi=600)
                plt.close(fig)

